# SHAP Analysis — XGBoost Feature Importance

**Standalone notebook.** Requires only `data_for_experiments.csv`.

Trains the best model (XGBoost, Full top-30 features) and computes SHAP values
to explain *which features drive predictions* and *how they drive them*.

SHAP (SHapley Additive exPlanations) replaces MDI (Mean Decrease Impurity)
as the feature importance method. MDI is biased toward high-cardinality features
and splits importance across correlated pairs. SHAP gives each feature a
consistent, model-agnostic contribution score per prediction.

**Figures produced:**
- `shap_bar_overall.png` — mean |SHAP| per feature across all classes
- `shap_beeswarm_strong_buy.png` — beeswarm for Strong Buy class
- `shap_beeswarm_strong_avoid.png` — beeswarm for Strong Avoid class
- `shap_category_importance.png` — importance grouped by feature category
- `shap_dependence_top6.png` — dependence plots for the 6 most important features
- `shap_heatmap.png` — SHAP value heatmap across test observations

In [ ]:
# Install shap if not already installed
# !pip install shap --quiet

from __future__ import annotations
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mtick
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

IN_FILE = Path('Models/Data/data_for_experiments.csv')
OUT_DIR = Path('Models/Figures')

TRAIN_CUTOFF = '2024-01'
FORWARD_DAYS = 7
MIN_COINS    = 20
TOP_N        = 30

LABEL_ORDER = ['Strong Avoid', 'Avoid', 'Neutral', 'Buy', 'Strong Buy']
LABEL_MAP   = {'Strong Avoid': 0, 'Avoid': 1, 'Neutral': 2, 'Buy': 3, 'Strong Buy': 4}
BINS        = [0, 0.10, 0.30, 0.70, 0.90, 1.001]

FEATURE_CATEGORIES = {
    'Volatility':  ['atr_pct', 'atr_14', 'volatility_30d', 'bb_width', 'bb_pct'],
    'Momentum':    ['return_1d', 'return_7d', 'return_30d', 'rsi_14',
                    'macd_hist', 'stoch_k', 'stoch_d'],
    'Trend':       ['price_vs_ema50', 'price_vs_ema200', 'ema_50_vs_200',
                    'adx', 'plus_di', 'minus_di'],
    'Volume/OBV':  ['volume', 'volume_vs_30d_avg', 'obv', 'obv_divergence'],
    'Price level': ['price_vs_ath', 'price_vs_atl', 'drawdown_from_90d_peak',
                    'coin_age_days'],
    'Market cap':  ['coin_mcap_share_recalc', 'coin_mcap_share_recalc_rank',
                    'market_cap_usd_zscore'],
    'Sentiment':   ['galaxy_score_zscore'],
}
CAT_COLORS = {
    'Volatility':  '#E24B4A',
    'Momentum':    '#1D9E75',
    'Trend':       '#378ADD',
    'Volume/OBV':  '#BA7517',
    'Price level': '#7F77DD',
    'Market cap':  '#888780',
    'Sentiment':   '#D4537E',
}

LEAKY_COLS = {
    'label', 'forward_return_7d', 'forward_sharpe_7d', 'forward_sharpe_rank',
    'return_1d_rank', 'return_1d_zscore', 'return_7d_rank', 'return_7d_zscore',
    'return_30d_rank', 'return_30d_zscore', 'volatility_30d_rank', 'volatility_30d_zscore',
    'rsi_14_rank', 'rsi_14_zscore', 'macd_hist_rank', 'macd_hist_zscore',
    'bb_pct_rank', 'bb_pct_zscore', 'atr_pct_rank', 'atr_pct_zscore',
    'obv_divergence_rank', 'obv_divergence_zscore', 'stoch_k_rank', 'stoch_k_zscore',
    'adx_rank', 'adx_zscore', 'volume_vs_30d_avg_rank', 'volume_vs_30d_avg_zscore',
    'drawdown_from_90d_peak_rank', 'drawdown_from_90d_peak_zscore',
    'price_vs_ath_rank', 'price_vs_ath_zscore', 'range_position_30d_rank',
    'range_position_30d_zscore', 'consecutive_up_days_rank', 'consecutive_up_days_zscore',
    'consecutive_down_days_rank', 'consecutive_down_days_zscore',
    'coin_age_days_rank', 'coin_age_days_zscore',
    'momentum_score', 'mean_reversion_score', 'trend_score',
    'asset_id', 'year_week', 'date', 'timestamp', 'exchange',
    'pair_symbol', 'source', 'open', 'high', 'low', 'close',
    'granularity', 'is_active',
}

CS_FEATURES = [
    'return_1d', 'return_7d', 'return_30d', 'volatility_30d',
    'rsi_14', 'macd_hist', 'bb_pct', 'atr_pct', 'obv_divergence',
    'stoch_k', 'adx', 'volume_vs_30d_avg', 'drawdown_from_90d_peak',
    'price_vs_ath', 'range_position_30d', 'consecutive_up_days',
    'consecutive_down_days', 'coin_age_days', 'galaxy_score', 'alt_rank',
    'market_cap_usd', 'coin_mcap_share_recalc', 'oi_usd', 'funding_rate',
    'taker_buy_ratio',
]

def prep(d, cols):
    return d[cols].fillna(0).replace([np.inf, -np.inf], 0).clip(-1e9, 1e9)

print('Config loaded. SHAP version:', shap.__version__)

## 1. Load data, build labels, train/test split

In [ ]:
df = pd.read_csv(IN_FILE, low_memory=False)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['asset_id', 'date']).reset_index(drop=True)
df['year_week'] = df['date'].dt.strftime('%G-W%V')   # must be before groupby
print(f'Loaded: {df.shape[0]:,} rows | {df["asset_id"].nunique()} coins')

def compute_forward_sharpe_7d(group):
    closes, results = group['close'].values, []
    for i in range(len(closes)):
        end = i + FORWARD_DAYS
        if end >= len(closes):
            results.append(np.nan); continue
        window     = closes[i:end]
        daily_rets = np.diff(window) / window[:-1]
        fwd_ret    = (window[-1] - window[0]) / window[0]
        vol        = daily_rets.std() * np.sqrt(365)
        results.append((fwd_ret / vol) if vol > 1e-8 else 0.0)
    return pd.Series(results, index=group.index)

print('Computing forward Sharpe...')
df['forward_sharpe_7d'] = (
    df.groupby('asset_id', group_keys=False).apply(compute_forward_sharpe_7d)
)
df['forward_return_7d'] = (
    df.groupby('asset_id')['close']
    .transform(lambda x: x.shift(-FORWARD_DAYS) / x - 1)
)

weekly = (
    df.dropna(subset=['forward_sharpe_7d'])
    .sort_values('date')
    .groupby(['asset_id', 'year_week']).last().reset_index()
)

def assign_labels(group):
    if len(group) < MIN_COINS:
        return group.assign(label=np.nan, forward_sharpe_rank=np.nan)
    q = group['forward_sharpe_7d'].rank(pct=True)
    group = group.copy()
    group['forward_sharpe_rank'] = q
    group['label'] = pd.cut(q, bins=BINS, labels=LABEL_ORDER, include_lowest=True)
    return group

weekly = (
    weekly.groupby('year_week', group_keys=False).apply(assign_labels)
    .dropna(subset=['label']).reset_index(drop=True)
)


if 'year_week' not in weekly.columns:
    weekly['year_week'] = pd.to_datetime(weekly['date']).dt.strftime('%G-W%V')

cs_cols = [c for c in CS_FEATURES if c in weekly.columns]
for col in cs_cols:
    weekly[f'{col}_rank']   = weekly.groupby('year_week')[col].transform(
        lambda x: x.rank(pct=True))
    weekly[f'{col}_zscore'] = weekly.groupby('year_week')[col].transform(
        lambda x: (x - x.mean()) / (x.std() if x.std() > 0 else 1))

cutoff_date = pd.Timestamp(TRAIN_CUTOFF + '-01')
train_df = weekly[pd.to_datetime(weekly['date']) <  cutoff_date].copy()
test_df  = weekly[pd.to_datetime(weekly['date']) >= cutoff_date].copy()
print(f'Train: {len(train_df):,} | Test: {len(test_df):,}')

all_features = [
    c for c in weekly.columns
    if c not in LEAKY_COLS
    and weekly[c].dtype in [np.float64, np.float32, np.int64, np.int32]
]
null_rates   = train_df[all_features].isnull().mean()
all_features = [f for f in all_features if null_rates[f] <= 0.30]

y_train = train_df['label'].map(LABEL_MAP)
y_test  = test_df['label'].map(LABEL_MAP)
print(f'Candidate features: {len(all_features)}')

## 2. Feature selection and train XGBoost

In [ ]:
print('RF pass-1 feature selection...')
rf_sel = RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=5,
                                 max_features='sqrt', class_weight='balanced',
                                 oob_score=True, n_jobs=-1, random_state=42)
rf_sel.fit(prep(train_df, all_features), y_train)
imp_series = pd.Series(rf_sel.feature_importances_, index=all_features).sort_values(ascending=False)
top30 = imp_series.head(TOP_N).index.tolist()
print(f'  OOB: {rf_sel.oob_score_*100:.2f}%')
print('Top-30 features selected:')
for i, f in enumerate(top30, 1):
    print(f'  {i:>2}. {f}')

# Train XGBoost on top-30
print('\nTraining XGBoost...')
model = XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='mlogloss', verbosity=0,
    random_state=42, n_jobs=-1
)
model.fit(prep(train_df, top30), y_train)
y_pred = model.predict(prep(test_df, top30))
acc = accuracy_score(y_test, y_pred)
print(f'Test accuracy: {acc*100:.2f}%  (vs 20% baseline, lift +{(acc-0.2)*100:.2f}pp)')

## 3. Compute SHAP values

In [ ]:
X_test    = prep(test_df, top30)
explainer = shap.TreeExplainer(model)
shap_raw  = explainer.shap_values(X_test)

if isinstance(shap_raw, list):
    shap_values = shap_raw
elif shap_raw.ndim == 3:
    shap_values = [shap_raw[:, :, i] for i in range(len(LABEL_ORDER))]
else:
    n_feats = len(top30)
    shap_values = [shap_raw[:, i*n_feats:(i+1)*n_feats] for i in range(len(LABEL_ORDER))]

CLASS_IDX = {'Strong Avoid':0,'Avoid':1,'Neutral':2,'Buy':3,'Strong Buy':4}

mean_abs_shap = pd.DataFrame(
    {label: np.abs(shap_values[idx]).mean(axis=0) for label, idx in CLASS_IDX.items()},
    index=top30
)
mean_abs_shap['overall'] = mean_abs_shap.mean(axis=1)
mean_abs_shap = mean_abs_shap.sort_values('overall', ascending=False)

print('SHAP values computed.')
print(f'  Shape per class: {shap_values[0].shape}')
print('\nTop-10 features by mean |SHAP| (overall):')
print(mean_abs_shap['overall'].head(10).round(4).to_string())

## 4. Figure 1 — Overall feature importance (mean |SHAP|)

In [ ]:
import tempfile, os

X_test = prep(test_df, top30)

print('Computing SHAP values (XGBoost 2.x compatibility fix)...')

with tempfile.NamedTemporaryFile(suffix='.ubj', delete=False) as tmp:
    tmp_path = tmp.name

model.get_booster().save_model(tmp_path)
booster_reload = model.get_booster().__class__()
import xgboost as xgb
booster_reload = xgb.Booster()
booster_reload.load_model(tmp_path)
os.unlink(tmp_path)

explainer = shap.TreeExplainer(booster_reload)
shap_raw  = explainer.shap_values(X_test.values)
if isinstance(shap_raw, list):
    shap_values = shap_raw
elif shap_raw.ndim == 3:
    shap_values = [shap_raw[:, :, i] for i in range(len(LABEL_ORDER))]
else:
    n_feats = len(top30)
    shap_values = [shap_raw[:, i*n_feats:(i+1)*n_feats] for i in range(len(LABEL_ORDER))]

CLASS_IDX = {'Strong Avoid':0,'Avoid':1,'Neutral':2,'Buy':3,'Strong Buy':4}

mean_abs_shap = pd.DataFrame(
    {label: np.abs(shap_values[idx]).mean(axis=0) for label, idx in CLASS_IDX.items()},
    index=top30
)
mean_abs_shap['overall'] = mean_abs_shap.mean(axis=1)
mean_abs_shap = mean_abs_shap.sort_values('overall', ascending=False)

print('SHAP values computed.')
print(f'  Shape per class: {shap_values[0].shape}')
print('\nTop-10 features by mean |SHAP| (overall):')
print(mean_abs_shap['overall'].head(10).round(4).to_string())

## 5. Figure 2 — Beeswarm: Strong Buy class

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

shap.summary_plot(
    shap_values[CLASS_IDX['Strong Buy']],
    X_test,
    feature_names=top30,
    max_display=30,
    show=False,
    plot_type='dot'
)

plt.title('SHAP beeswarm — Strong Buy class\n'
          '(right = pushes toward Strong Buy, left = pushes away)',
          fontsize=11, fontweight='500', pad=10)
plt.tight_layout()
p = OUT_DIR / 'shap_beeswarm_strong_buy.png'
plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); print(f'Saved -> {p.name}')

## 6. Figure 3 — Beeswarm: Strong Avoid class

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

shap.summary_plot(
    shap_values[CLASS_IDX['Strong Avoid']],
    X_test,
    feature_names=top30,
    max_display=30,
    show=False,
    plot_type='dot'
)

plt.title('SHAP beeswarm — Strong Avoid class\n'
          '(right = pushes toward Strong Avoid, left = pushes away)',
          fontsize=11, fontweight='500', pad=10)
plt.tight_layout()
p = OUT_DIR / 'shap_beeswarm_strong_avoid.png'
plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); print(f'Saved -> {p.name}')

## 7. Figure 4 — Category-level SHAP importance

In [ ]:
cat_shap = {}
for cat, feats in FEATURE_CATEGORIES.items():
    present = [f for f in feats if f in mean_abs_shap.index]
    if present:
        cat_shap[cat] = mean_abs_shap.loc[present, 'overall'].sum()

cat_df = pd.Series(cat_shap).sort_values(ascending=True)
cat_colors_plot = [CAT_COLORS[c] for c in cat_df.index]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor('white')
ax = axes[0]
ax.barh(range(len(cat_df)), cat_df.values,
        color=cat_colors_plot, edgecolor='white', linewidth=0.5, alpha=0.9)
ax.set_yticks(range(len(cat_df)))
ax.set_yticklabels(cat_df.index, fontsize=11)
ax.set_xlabel('Total mean |SHAP value|', fontsize=10)
ax.set_title('SHAP importance by feature category',
             fontsize=11, fontweight='500', pad=10)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')
for i, val in enumerate(cat_df.values):
    ax.text(val + 0.0002, i, f'{val:.4f}', va='center', fontsize=9)

ax = axes[1]
cat_class_df = pd.DataFrame({
    label: {
        cat: mean_abs_shap.loc[
            [f for f in feats if f in mean_abs_shap.index], label
        ].sum()
        for cat, feats in FEATURE_CATEGORIES.items()
    }
    for label, _ in CLASS_IDX.items()
})

cmap = mcolors.LinearSegmentedColormap.from_list('shap', ['#E6F1FB', '#185FA5', '#042C53'])
im = ax.imshow(cat_class_df.values, cmap=cmap, aspect='auto')
ax.set_xticks(range(len(LABEL_ORDER)))
ax.set_xticklabels(LABEL_ORDER, rotation=20, ha='right', fontsize=9)
ax.set_yticks(range(len(FEATURE_CATEGORIES)))
ax.set_yticklabels(list(FEATURE_CATEGORIES.keys()), fontsize=10)
plt.colorbar(im, ax=ax, shrink=0.8, label='Mean |SHAP|')
ax.set_title('SHAP importance: category × predicted class',
             fontsize=11, fontweight='500', pad=10)

for i in range(len(FEATURE_CATEGORIES)):
    for j in range(len(LABEL_ORDER)):
        val = cat_class_df.values[i, j]
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=8,
                color='white' if val > cat_class_df.values.max() * 0.6 else '#2C2C2A')

plt.tight_layout()
p = OUT_DIR / 'shap_category_importance.png'
plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); print(f'Saved -> {p.name}')

## 8. Figure 5 — Dependence plots: top-6 features

In [ ]:
top6 = mean_abs_shap['overall'].sort_values(ascending=False).head(6).index.tolist()
print('Top-6 features for dependence plots:', top6)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.patch.set_facecolor('white')
axes = axes.flatten()

SB_IDX = CLASS_IDX['Strong Buy']

for ax, feat in zip(axes, top6):
    if feat not in X_test.columns:
        continue
    feat_idx  = list(X_test.columns).index(feat)
    feat_vals = X_test[feat].values
    shap_vals = shap_values[SB_IDX][:, feat_idx]
    sc = ax.scatter(feat_vals, shap_vals,
                    c=feat_vals, cmap='RdYlGn', alpha=0.4,
                    s=6, linewidths=0)
    ax.axhline(0, color='#888780', lw=0.8, ls='--')
    try:
        sorted_idx = np.argsort(feat_vals)
        xsort = feat_vals[sorted_idx]
        ysort = shap_vals[sorted_idx]
        window = max(50, len(xsort)//20)
        y_smooth = pd.Series(ysort).rolling(window, center=True, min_periods=1).mean().values
        ax.plot(xsort, y_smooth, color='#1D3A6E', lw=1.8, alpha=0.9)
    except Exception:
        pass

    ax.set_xlabel(feat, fontsize=9)
    ax.set_ylabel('SHAP value (Strong Buy)', fontsize=9)
    cat_name = next((c for c, fs in FEATURE_CATEGORIES.items()
                     if feat.replace('_rank','').replace('_zscore','') in fs), 'Other')
    ax.set_title(f'{feat}\n[{cat_name}]', fontsize=9, fontweight='500', pad=6)
    ax.spines[['top','right']].set_visible(False)
    ax.set_facecolor('white')
    plt.colorbar(sc, ax=ax, shrink=0.7, label='Feature value')

plt.suptitle('SHAP dependence plots — top-6 features (Strong Buy class)\n'
             'Upward slope = higher feature value → stronger Strong Buy prediction',
             fontsize=12, fontweight='500', y=1.01)
plt.tight_layout()
p = OUT_DIR / 'shap_dependence_top6.png'
plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); print(f'Saved -> {p.name}')

## 9. Figure 6 — MDI vs SHAP: do they agree?

In [ ]:
def get_feature_color(feat):
    base = feat.replace('_rank','').replace('_zscore','')
    for cat, feats in FEATURE_CATEGORIES.items():
        if base in feats:
            return CAT_COLORS[cat], cat
    return '#B4B2A9', 'Other'

mdi_imp   = pd.Series(model.feature_importances_, index=top30)
shap_imp  = mean_abs_shap['overall']

mdi_rank  = mdi_imp.rank(ascending=False)
shap_rank = shap_imp.rank(ascending=False)

rank_df = pd.DataFrame({'MDI rank': mdi_rank, 'SHAP rank': shap_rank}).sort_values('SHAP rank')
rank_df['rank_diff'] = (rank_df['MDI rank'] - rank_df['SHAP rank']).abs()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')

ax = axes[0]
colors_scatter = [get_feature_color(f)[0] for f in rank_df.index]
ax.scatter(rank_df['MDI rank'], rank_df['SHAP rank'],
           c=colors_scatter, s=80, alpha=0.85, edgecolors='white', linewidths=0.5)
ax.plot([1, TOP_N], [1, TOP_N], color='#888780', lw=1.0, ls='--', label='Perfect agreement')
for feat, row in rank_df.iterrows():
    if row['rank_diff'] >= 8:
        ax.annotate(feat, (row['MDI rank'], row['SHAP rank']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points', alpha=0.8)
ax.set_xlabel('MDI rank (1 = most important)', fontsize=10)
ax.set_ylabel('SHAP rank (1 = most important)', fontsize=10)
ax.set_title('MDI rank vs SHAP rank\n(points near diagonal = agreement)',
             fontsize=10, fontweight='500', pad=8)
ax.legend(fontsize=8, framealpha=0)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')

ax = axes[1]
top_disagreements = rank_df.nlargest(15, 'rank_diff')
bar_cols = [get_feature_color(f)[0] for f in top_disagreements.index]
y = range(len(top_disagreements))
ax.barh(y, top_disagreements['rank_diff'],
        color=bar_cols, edgecolor='white', linewidth=0.5, alpha=0.9)
ax.set_yticks(y)
ax.set_yticklabels(top_disagreements.index, fontsize=9)
ax.set_xlabel('|MDI rank − SHAP rank|', fontsize=10)
ax.set_title('Top-15 MDI vs SHAP disagreements\n(large value = MDI and SHAP disagree most)',
             fontsize=10, fontweight='500', pad=8)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')

from scipy.stats import spearmanr
rho, pval = spearmanr(rank_df['MDI rank'], rank_df['SHAP rank'])
print(f'Spearman correlation between MDI and SHAP rankings: ρ = {rho:.3f}  (p = {pval:.4f})')
if rho > 0.8:
    print('  High agreement — MDI and SHAP largely agree on feature ranking.')
elif rho > 0.6:
    print('  Moderate agreement — some features ranked differently between methods.')
else:
    print('  Low agreement — MDI importances may be biased; SHAP is more reliable.')

plt.tight_layout()
p = OUT_DIR / 'shap_vs_mdi.png'
plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved -> {p.name}')


mdi_imp = pd.Series(model.feature_importances_, index=top30)
shap_imp = mean_abs_shap['overall']

mdi_rank  = mdi_imp.rank(ascending=False)
shap_rank = shap_imp.rank(ascending=False)

rank_df = pd.DataFrame({'MDI rank': mdi_rank, 'SHAP rank': shap_rank}).sort_values('SHAP rank')
rank_df['rank_diff'] = (rank_df['MDI rank'] - rank_df['SHAP rank']).abs()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor('white')

# Left: scatter MDI rank vs SHAP rank
ax = axes[0]
colors_scatter = [get_feature_color(f)[0] for f in rank_df.index]
ax.scatter(rank_df['MDI rank'], rank_df['SHAP rank'],
           c=colors_scatter, s=80, alpha=0.85, edgecolors='white', linewidths=0.5)
ax.plot([1, TOP_N], [1, TOP_N], color='#888780', lw=1.0, ls='--', label='Perfect agreement')
for feat, row in rank_df.iterrows():
    if row['rank_diff'] >= 8:  # annotate large disagreements
        ax.annotate(feat, (row['MDI rank'], row['SHAP rank']),
                    fontsize=7, xytext=(4, 4), textcoords='offset points', alpha=0.8)
ax.set_xlabel('MDI rank (1 = most important)', fontsize=10)
ax.set_ylabel('SHAP rank (1 = most important)', fontsize=10)
ax.set_title('MDI rank vs SHAP rank\n(points near diagonal = agreement)',
             fontsize=10, fontweight='500', pad=8)
ax.legend(fontsize=8, framealpha=0)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')

ax = axes[1]
top_disagreements = rank_df.nlargest(15, 'rank_diff')
bar_cols = [get_feature_color(f)[0] for f in top_disagreements.index]
y = range(len(top_disagreements))
ax.barh(y, top_disagreements['rank_diff'],
        color=bar_cols, edgecolor='white', linewidth=0.5, alpha=0.9)
ax.set_yticks(y)
ax.set_yticklabels(top_disagreements.index, fontsize=9)
ax.set_xlabel('|MDI rank − SHAP rank|', fontsize=10)
ax.set_title('Top-15 MDI vs SHAP disagreements\n(large value = MDI and SHAP disagree most)',
             fontsize=10, fontweight='500', pad=8)
ax.spines[['top','right']].set_visible(False)
ax.set_facecolor('white')

# Spearman correlation
from scipy.stats import spearmanr
rho, pval = spearmanr(rank_df['MDI rank'], rank_df['SHAP rank'])
print(f'Spearman correlation between MDI and SHAP rankings: ρ = {rho:.3f}  (p = {pval:.4f})')
if rho > 0.8:
    print('  High agreement — MDI and SHAP largely agree on feature ranking.')
elif rho > 0.6:
    print('  Moderate agreement — some features ranked differently between methods.')
else:
    print('  Low agreement — MDI importances may be biased; SHAP is more reliable.')

plt.tight_layout()
p = OUT_DIR / 'shap_vs_mdi.png'
plt.savefig(p, dpi=150, bbox_inches='tight', facecolor='white')
plt.show(); print(f'Saved -> {p.name}')

## 10. Save SHAP importance table and print summary

In [ ]:
mean_abs_shap['category'] = mean_abs_shap.index.map(
    lambda f: next((c for c, fs in FEATURE_CATEGORIES.items()
                    if f.replace('_rank','').replace('_zscore','') in fs), 'Other')
)
mean_abs_shap['mdi_importance'] = mdi_imp
mean_abs_shap['shap_rank'] = mean_abs_shap['overall'].rank(ascending=False).astype(int)
mean_abs_shap['mdi_rank']  = mdi_imp.rank(ascending=False).astype(int)

mean_abs_shap.to_csv(DYPLOM / 'shap_importance_table.csv')

print('='*70)
print('SHAP IMPORTANCE SUMMARY — XGBoost top-30 features')
print('='*70)
print(f'{"Feature":<35} {"Category":<14} {"SHAP":>8}  {"S-rank":>6}  {"MDI-rank":>8}')
print('-'*75)
for feat, row in mean_abs_shap.sort_values('overall', ascending=False).iterrows():
    print(f'{feat:<35} {row["category"]:<14} {row["overall"]:>8.5f}  '
          f'{row["shap_rank"]:>6.0f}  {row["mdi_rank"]:>8.0f}')

print('\n=== Category totals (SHAP) ===')
cat_totals = mean_abs_shap.groupby('category')['overall'].sum().sort_values(ascending=False)
cat_pct    = cat_totals / cat_totals.sum() * 100
for cat, total in cat_totals.items():
    print(f'  {cat:<14} {total:.4f}  ({cat_pct[cat]:.1f}%)')

print('\nSaved -> shap_importance_table.csv')
print('\n=== Output figures ===')
for f in sorted(OUT_DIR.iterdir()):
    if 'shap' in f.name:
        print(f'  {f.name}')